<a href="https://colab.research.google.com/github/taruchit/Using-RAG-for-Planning-stay-in-Rajasthan/blob/Use-cases/RAG_Rajasthan_Tourism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -qU langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [11]:
import re
import os
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [12]:
from google.colab import userdata

# Securely retrieve the OpenAI API key from Colab Secrets
try:
    openai_api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    # Fallback to os.environ if not found in Colab Secrets
    openai_api_key = os.environ.get("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("Please set your 'OPENAI_API_KEY' in the Colab Secrets (the key icon on the left sidebar).")

In [13]:
# 1. Load the generated text file and preserve page metadata
with open("tariff_summary.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

pages = re.split(r"(?=== Page \d+ ===)", raw_text)
raw_docs = []

for section in pages:
  section = section.strip()
  if not section:
    continue
  match = re.match(r"=== Page (\d+) ===", section)
  page_num = int(match.group(1)) if match else None
  raw_docs.append(Document(page_content=section, metadata={"page": page_num}))

In [14]:
# 2. Chunk while keeping table rows and page boundaries intact
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    separators=["\n=== Page ", "\n\n", "\n", " "],
)
chunks = splitter.split_documents(raw_docs)
print(f"Created {len(chunks)} chunks from {len(raw_docs)} pages.")

Created 39 chunks from 16 pages.


In [16]:
# 3. Embed and index into FAISS
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=openai_api_key)
vectorstore = FAISS.from_documents(chunks, embeddings)
# Optional: save locally so you don't re-embed every run
vectorstore.save_local("tariff_faiss_index")

In [17]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [18]:
# 4. Build the grounded Q&A chain
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a tariff assistant. Answer the user's question concisely using ONLY the "
        "provided context from the tariff document. Quote exact tariff codes, "
        "rates, and conditions, and cite the page number(s). If the answer is "
        "not in the context, say you cannot find it in the document.\n\n"
        "Context:\n{context}",
    ),
    ("human", "{question}"),
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=openai_api_key)

In [19]:
def format_docs(docs):
  return "\n\n---\n\n".join(
      f"[Page {d.metadata.get('page', '?')}]\n{d.page_content}" for d in docs
  )

In [20]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Ask questions
question = "What are the applicable tariff rates in Ajmer?"
print(rag_chain.invoke(question))

The applicable tariff rates in Ajmer for Hotel Khadim are as follows:

- **Double Room:**
  - Suite: ₹3000
  - Sup Dlx: ₹2500
  - Dlx: ₹1800

- **Single Room:**
  - Suite: ₹2700
  - Sup Dlx: ₹2200
  - Dlx: ₹1500

- **Extra Bed:**
  - Suite: ₹500
  - Sup Dlx: ₹500
  - Dlx: ₹400
  - A.C.: ₹300

These rates are applicable from April to March (Page 3).


In [21]:
question = "List of hotels available in Jaisalmer?"
print(rag_chain.invoke(question))

The hotels available in Jaisalmer are:

1. **Hotel: Sam Dhani**
   - **Tariff:** CP Tariff
   - **Rooms:** 3 (Tent), 10 (Non A.C.), 3 (Tent), 10 (Non A.C.)
   - **Double Rates:** 2750, 1500, 4750, 4000
   - **Single Rates:** 2100, 1100, 3750, 3000
   - **E. Bed Rates:** 350, 300, 150, 150

2. **Hotel: Moomal**
   - **Tariff:** CP Tariff
   - **Rooms:** 17 (A.C.), 21 (Non A.C. Family Room)
   - **Double Rates:** 1500, 1000, 1500, 2400, 1400, 2000
   - **Single Rates:** 1200, 800, 1500, 2000, 1000, 2000
   - **E. Bed Rates:** 350, 300, 300, 500, 400, 400

(Information sourced from the tariff document, Page None)


In [22]:
# Question beyond scope of the document
question = "What are the applicable tariff rates in Bangalore?"
print(rag_chain.invoke(question))

I cannot find it in the document.
